# Week 3 — Public-Data Pipeline & Warehouse QA

Runs the ingestion pipeline, profiles the DuckDB warehouse, and charts a few baseline trends that feed Week 4 market sizing. All 12 sources are public (real APIs documented in `src/ingest/datasets.py`); offline fallbacks carry the real schema so this runs end-to-end without network.

## 1. Run the pipeline + build the warehouse
(Idempotent. Set `INGEST_ALLOW_NETWORK=1` in the environment to pull live APIs.)

In [ ]:
import subprocess, sys
for mod in ['src.ingest.run_all','src.ingest.build_warehouse','src.ingest.quality_report']:
    print('running', mod)
    print(subprocess.run([sys.executable,'-m',mod], capture_output=True, text=True, cwd='..').stdout[-400:])

## 2. Inventory — schemas & tables

In [ ]:
import duckdb, pandas as pd
con = duckdb.connect('../data/week03/analytics.duckdb', read_only=True)
inv = con.execute('''SELECT table_schema, table_name FROM information_schema.tables
   WHERE table_schema NOT IN ('information_schema','pg_catalog','main') ORDER BY 1,2''').df()
print(inv.to_string(index=False))

## 3. Null-rate & coverage profile across all tables

In [ ]:
rows=[]
for s,t in con.execute('''SELECT table_schema,table_name FROM information_schema.tables
   WHERE table_schema NOT IN ('information_schema','pg_catalog','main')
     AND table_type='BASE TABLE' AND table_name NOT LIKE '\_%' ESCAPE '\' ''').fetchall():
    df=con.execute(f'SELECT * FROM {s}."{t}"').df()
    yr=''
    if 'year' in df.columns:
        y=pd.to_numeric(df['year'],errors='coerce').dropna()
        yr=f'{int(y.min())}-{int(y.max())}' if len(y) else ''
    rows.append({'schema':s,'table':t,'rows':len(df),'cols':df.shape[1],
                 'max_null_%':round(float((df.isna().mean()*100).max()),1),'years':yr})
prof=pd.DataFrame(rows); print(prof.to_string(index=False))
assert prof['rows'].min()>0, 'a table is empty'
print('\nTotal rows:', int(prof['rows'].sum()), '| tables:', len(prof))

## 4. Baseline trends for Week 4 sizing

In [ ]:
import matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,3,figsize=(14,4))
g=con.execute('SELECT year,employment FROM indoor_security.bls_security_guards ORDER BY year').df()
ax[0].plot(g['year'],g['employment'],marker='o'); ax[0].set_title('US security guards (BLS)'); ax[0].set_xlabel('year')
h=con.execute('SELECT year,employment FROM eldercare.bls_home_health_aides ORDER BY year').df()
ax[1].plot(h['year'],h['employment'],marker='o',color='seagreen'); ax[1].set_title('US home health aides (BLS)'); ax[1].set_xlabel('year')
r=con.execute('SELECT year,works_count FROM humanoid.openalex_robotics_publications ORDER BY year').df()
ax[2].plot(r['year'],r['works_count'],marker='o',color='indianred'); ax[2].set_title('Robotics publications (OpenAlex)'); ax[2].set_xlabel('year')
plt.tight_layout(); plt.savefig('week03_baseline_trends.png',dpi=120); print('saved week03_baseline_trends.png')

## 5. Country demand context (eldercare)
Population 65+ vs R&D intensity — a starting frame for Week 4 TAM by geography.

In [ ]:
ctx=con.execute('SELECT * FROM shared.v_country_context ORDER BY pop_65plus_pct DESC').df()
print(ctx.to_string(index=False))
con.close()

## Notes
- Provenance for every table (source, URL, license, sha256, retrieval date, run mode) is in `data/week03/ingest_manifest.jsonl` and summarized in `reports/week03/data_quality_report.html`.
- To refresh from live sources: `INGEST_ALLOW_NETWORK=1 python -m src.ingest.run_all` then rebuild the warehouse.
- These cleaned, versioned tables are the inputs for **Week 4** (TAM/SAM/SOM).